# PESO SCAN — experimental Roboflow denomination candidate

This notebook converts the downloaded YOLOv8 banknote detection dataset into note crops, trains a MobileNetV3 Small six-denomination classifier, evaluates the dataset's supplied test split, and exports an ONNX candidate. It does not detect counterfeits or authenticate notes.

**Important data limitation:** this Roboflow export has image-level train/valid/test folders and augmented copies, but no physical-banknote IDs. This notebook checks that Roboflow augmented siblings stay in one split; it cannot prove that photos of the same physical note do not occur in different splits. The output is marked `deployment_approved: false` and must be treated as an experimental candidate.

The dataset states CC BY 4.0. Keep attribution to the Roboflow Philippine Banknotes dataset with any use, and review the dataset's license and photo rights before commercial distribution: https://universe.roboflow.com/philippine-banknotes/philippine-banknotes/dataset/8

## Before running
In Google Drive, create a viewer link for the ZIP (**General access: Anyone with the link**) and paste that link when the first cell asks. This is a public Roboflow dataset; do not use this method for private files. In Colab, choose **Runtime → Change runtime type → T4 GPU** if available; free GPU availability varies. This notebook downloads the ZIP directly and does not mount Drive or upload the large file through the browser.

In [ ]:
from pathlib import Path
import subprocess, sys
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'gdown'])
import gdown
ZIP_NAME = 'Philippine Banknotes.v8i.yolov8.zip'
ZIP_PATH = Path('/content') / ZIP_NAME
ZIP_URL = input('Paste the Google Drive ZIP file link (not its folder link): ').strip()
if '/folders/' in ZIP_URL: raise ValueError('This is a folder link. Open the folder in Drive, right-click the ZIP file, choose Share, and paste the ZIP file link here.')
if not ZIP_URL.startswith('https://drive.google.com/'): raise ValueError('Use the Google Drive share link copied from the ZIP file.')
downloaded = gdown.download(url=ZIP_URL, output=str(ZIP_PATH), quiet=False, fuzzy=True)
if not downloaded or not ZIP_PATH.is_file() or ZIP_PATH.stat().st_size < 100_000_000:
    raise RuntimeError('The ZIP download did not complete. Check that Drive access is set to Anyone with the link (Viewer), then rerun this cell.')
DRIVE_MOUNTED = False
print(f'Dataset archive ready ({ZIP_PATH.stat().st_size / (1024**2):.0f} MiB).')

In [ ]:
import subprocess, sys
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'PyYAML', 'onnx', 'onnxruntime', 'scikit-learn'])
import torch, torchvision
print('PyTorch:', torch.__version__, '| TorchVision:', torchvision.__version__)
print('Device:', 'CUDA GPU' if torch.cuda.is_available() else 'CPU (slower)')

In [ ]:
from pathlib import Path
import hashlib, json, re, shutil, zipfile
if 'ZIP_PATH' not in globals(): raise RuntimeError('Dataset ZIP path is not set. Run the first download cell, then rerun this cell.')
if not ZIP_PATH.is_file(): raise FileNotFoundError(f'Dataset ZIP is missing at {ZIP_PATH}. Rerun the first download cell.')
import yaml
WORK = Path('/content/peso_scan_candidate')
EXTRACT = WORK / 'dataset'
if EXTRACT.exists(): shutil.rmtree(EXTRACT)
EXTRACT.mkdir(parents=True)
with zipfile.ZipFile(ZIP_PATH) as archive:
    root = EXTRACT.resolve()
    for member in archive.infolist():
        target = (EXTRACT / member.filename).resolve()
        if target != root and root not in target.parents:
            raise ValueError('Unsafe path found in dataset ZIP')
    archive.extractall(EXTRACT)
yaml_files = list(EXTRACT.rglob('data.yaml'))
if len(yaml_files) != 1: raise RuntimeError(f'Expected one data.yaml, found {len(yaml_files)}')
DATASET = yaml_files[0].parent
config = yaml.safe_load(yaml_files[0].read_text(encoding='utf-8'))
raw_names = config['names']
names = raw_names if isinstance(raw_names, list) else [raw_names[str(i)] for i in range(len(raw_names))]
print('Dataset root:', DATASET)
print('Classes:', names)
assert all((DATASET / split / 'images').is_dir() and (DATASET / split / 'labels').is_dir() for split in ('train', 'valid', 'test')), 'Expected train/valid/test images and labels folders.'

## Convert labeled banknote boxes to denomination crops

Coin classes are excluded. Paper/polymer ₱1,000 classes are combined into the ₱1,000 denomination. The supplied splits are preserved. Augmentation siblings are grouped by their shared Roboflow filename stem and checked for split overlap. This is a source-image check, not physical-note identity verification.

In [ ]:
from pathlib import Path
from PIL import Image
from collections import defaultdict, Counter
import csv, hashlib, os, re, shutil
missing_setup = [key for key in ('DATASET', 'WORK', 'names') if key not in globals()]
if missing_setup: raise RuntimeError(f'Missing {missing_setup}. Run the dataset extraction cell first, then rerun crop preparation.')
DENOMS = (20, 50, 100, 200, 500, 1000)
DENOM_INDEX = {value: i for i, value in enumerate(DENOMS)}
def normalize_name(value): return re.sub(r'[^a-z0-9]+', '_', value.lower()).strip('_')
class_to_denom = {}
for class_id, class_name in enumerate(names):
    key = normalize_name(class_name)
    match = re.search(r'(1000|500|200|100|50|20)_pesos?$', key)
    if match: class_to_denom[class_id] = int(match.group(1))
print('Included banknote classes:', {names[k]: v for k, v in class_to_denom.items()})
if set(class_to_denom.values()) != set(DENOMS): raise RuntimeError('The dataset does not include all six target denominations.')
CROPS = WORK / 'crops'
if CROPS.exists(): shutil.rmtree(CROPS)
CROPS.mkdir(parents=True)
rows, group_splits, counts = [], defaultdict(set), Counter()
image_exts = {'.jpg', '.jpeg', '.png', '.bmp', '.webp'}
# Assign every Roboflow filename group to exactly one split before cropping.
# If augmented siblings leaked across splits, training takes precedence; held-out copies are skipped.
original_group_splits = defaultdict(set)
for source_split in ('train', 'valid', 'test'):
    for source_image in sorted((DATASET / source_split / 'images').iterdir()):
        if source_image.suffix.lower() not in image_exts: continue
        source_group = re.sub(r'\.rf\.[0-9a-fA-F]{16,}$', '', source_image.stem)
        original_group_splits[source_group].add(source_split)
overlap = {g: sorted(s) for g, s in original_group_splits.items() if len(s) > 1}
group_assignment = {g: ('train' if 'train' in s else 'valid' if 'valid' in s else 'test') for g, s in original_group_splits.items()}
print('Overlapping Roboflow groups reassigned to one split:', len(overlap), '| rule: train > valid > test')
for split in ('train', 'valid', 'test'):
    for image_path in sorted((DATASET / split / 'images').iterdir()):
        if image_path.suffix.lower() not in image_exts: continue
        label_path = DATASET / split / 'labels' / f'{image_path.stem}.txt'
        if not label_path.is_file(): continue
        source_group = re.sub(r'\.rf\.[0-9a-fA-F]{16,}$', '', image_path.stem)
        if split != group_assignment[source_group]: continue
        group_splits[source_group].add(split)
        with Image.open(image_path) as opened: image = opened.convert('RGB')
        width, height = image.size
        for object_index, line in enumerate(label_path.read_text().splitlines()):
            parts = line.split()
            if len(parts) < 5: continue
            class_id = int(float(parts[0]))
            if class_id not in class_to_denom: continue
            cx, cy, bw, bh = map(float, parts[1:5])
            x1, y1 = (cx-bw/2)*width, (cy-bh/2)*height
            x2, y2 = (cx+bw/2)*width, (cy+bh/2)*height
            pad_x, pad_y = (x2-x1)*0.04, (y2-y1)*0.04
            box = (max(0, int(x1-pad_x)), max(0, int(y1-pad_y)), min(width, int(x2+pad_x)), min(height, int(y2+pad_y)))
            if box[2] <= box[0] or box[3] <= box[1]: continue
            denom = class_to_denom[class_id]
            class_dir = f'php_{denom}'
            out_dir = CROPS / split / class_dir
            out_dir.mkdir(parents=True, exist_ok=True)
            unique = hashlib.sha1(str(image_path.relative_to(DATASET)).encode()).hexdigest()[:12]
            out_path = out_dir / f'{unique}_{object_index}.jpg'
            image.crop(box).save(out_path, quality=95)
            rel_path = out_path.relative_to(CROPS).as_posix()
            digest = hashlib.sha256(out_path.read_bytes()).hexdigest()
            rows.append({'image_path': rel_path, 'class_id': denom, 'source_group': source_group, 'split': split, 'sha256': digest})
            counts[(split, denom)] += 1
remaining_overlap = {g: sorted(s) for g, s in group_splits.items() if len(s) > 1}
if remaining_overlap: raise RuntimeError(f'Internal grouping error: {len(remaining_overlap)} groups still cross splits.')
for split in ('train', 'valid', 'test'):
    missing = [d for d in DENOMS if counts[(split, d)] == 0]
    if missing: raise RuntimeError(f'{split} split is missing target classes: {missing}')
manifest_path = WORK / 'candidate_manifest.csv'
with manifest_path.open('w', newline='') as f:
    writer = csv.DictWriter(f, fieldnames=['image_path','class_id','source_group','split','sha256'])
    writer.writeheader(); writer.writerows(rows)
print('Crops:', len(rows), '| split/class counts:', dict(counts))
print('Source-image groups checked:', len(group_splits), '| remaining cross-split groups:', len(remaining_overlap))

In [ ]:
from pathlib import Path
from collections import Counter
from PIL import Image
import csv
import torch, torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms
missing_setup = [key for key in ('rows', 'CROPS', 'manifest_path', 'DENOMS') if key not in globals()]
if missing_setup: raise RuntimeError(f'Missing {missing_setup}. Run crop preparation first, then rerun training.')
def safe_pad(image, size=(224, 224)):
    width, height = image.size
    if width < 1 or height < 1: raise ValueError(f'Cannot resize empty image with size {image.size}')
    scale = min(size[0] / width, size[1] / height)
    new_width = min(size[0], max(1, round(width * scale)))
    new_height = min(size[1], max(1, round(height * scale)))
    resized = image.resize((new_width, new_height), Image.Resampling.LANCZOS)
    canvas = Image.new('RGB', size, (0, 0, 0))
    canvas.paste(resized, ((size[0] - new_width) // 2, (size[1] - new_height) // 2))
    return canvas
# Check every generated crop before DataLoader workers try to resize it.
valid_rows, invalid_crops = [], []
for row in rows:
    crop_path = CROPS / row['image_path']
    try:
        with Image.open(crop_path) as crop:
            if crop.width < 2 or crop.height < 2: raise ValueError(f'degenerate crop dimensions: {crop.size}')
            if max(crop.size) / min(crop.size) > 12: raise ValueError(f'implausibly thin crop dimensions: {crop.size}')
            crop.verify()
        valid_rows.append(row)
    except Exception as error:
        invalid_crops.append((row, str(error)))
if invalid_crops:
    print('Skipping unreadable/empty generated crops:', len(invalid_crops))
    rows = valid_rows
    counts = Counter((r['split'], r['class_id']) for r in rows)
    for split in ('train', 'valid', 'test'):
        missing = [d for d in DENOMS if counts[(split, d)] == 0]
        if missing: raise RuntimeError(f'After removing invalid crops, {split} is missing classes: {missing}')
    with manifest_path.open('w', newline='') as f:
        writer = csv.DictWriter(f, fieldnames=['image_path','class_id','source_group','split','sha256'])
        writer.writeheader(); writer.writerows(rows)
    for row, error in invalid_crops[:10]: print(row['image_path'], error)
CLASS_TO_INDEX = {value: index for index, value in enumerate(DENOMS)}
MEAN, STD = (0.485, 0.456, 0.406), (0.229, 0.224, 0.225)
class CropDataset(Dataset):
    def __init__(self, split, training=False):
        self.rows = [r for r in rows if r['split'] == split]
        self.training = training
        if not self.rows: raise ValueError(f'No crops in {split}')
        self.to_tensor = transforms.ToTensor()
        self.normalize = transforms.Normalize(MEAN, STD)
        self.jitter = transforms.ColorJitter(brightness=0.12, contrast=0.12, saturation=0.08)
    def __len__(self): return len(self.rows)
    def __getitem__(self, index):
        row = self.rows[index]
        image_path = CROPS / row['image_path']
        with Image.open(image_path) as opened: image = opened.convert('RGB')
        if image.width < 2 or image.height < 2: raise ValueError(f'Degenerate crop reached training: {row["image_path"]} size={image.size}')
        if max(image.size) / min(image.size) > 12: raise ValueError(f'Implausibly thin crop reached training: {row["image_path"]} size={image.size}')
        image = safe_pad(image, (224,224))
        if self.training:
            if torch.rand(()) < 0.5: image = image.transpose(Image.Transpose.FLIP_LEFT_RIGHT)
            image = self.jitter(image)
        x = self.normalize(self.to_tensor(image))
        return x, CLASS_TO_INDEX[int(row['class_id'])]
train_ds, val_ds = CropDataset('train', True), CropDataset('valid')
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
batch_size = 32 if device.type == 'cuda' else 8
train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True, num_workers=0, pin_memory=device.type=='cuda')
val_loader = DataLoader(val_ds, batch_size=batch_size, shuffle=False, num_workers=0, pin_memory=device.type=='cuda')
model = models.mobilenet_v3_small(weights=None)
model.classifier[-1] = nn.Linear(model.classifier[-1].in_features, len(DENOMS))
model = model.to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4, weight_decay=0.01)
loss_fn = nn.CrossEntropyLoss()
scaler = torch.amp.GradScaler('cuda', enabled=device.type=='cuda')
def run_epoch(loader, training):
    model.train(training); loss_sum = correct = seen = 0
    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)
        if training: optimizer.zero_grad(set_to_none=True)
        with torch.set_grad_enabled(training):
            with torch.autocast(device_type=device.type, enabled=device.type=='cuda'):
                logits = model(images); loss = loss_fn(logits, labels)
            if training:
                scaler.scale(loss).backward(); scaler.step(optimizer); scaler.update()
        count = labels.size(0); loss_sum += loss.item()*count; correct += (logits.argmax(1)==labels).sum().item(); seen += count
    return loss_sum/max(seen,1), correct/max(seen,1)
best_acc, best_state = -1, None
EPOCHS = 12
for epoch in range(1, EPOCHS+1):
    tr_loss, tr_acc = run_epoch(train_loader, True)
    va_loss, va_acc = run_epoch(val_loader, False)
    print(f'Epoch {epoch:02d}/{EPOCHS} | train loss {tr_loss:.4f} acc {tr_acc:.3f} | val loss {va_loss:.4f} acc {va_acc:.3f}')
    if va_acc > best_acc:
        best_acc = va_acc
        best_state = {k: v.detach().cpu().clone() for k,v in model.state_dict().items()}
if best_state is None: raise RuntimeError('Training did not produce a checkpoint')
model.load_state_dict(best_state); model = model.cpu().eval()
CHECKPOINT = WORK / 'candidate_checkpoint.pt'
torch.save({'state_dict': best_state, 'class_ids': list(DENOMS), 'architecture':'torchvision.mobilenet_v3_small'}, CHECKPOINT)
print('Best validation accuracy:', best_acc)

## Held-out test evaluation and ONNX export

The test split is evaluated once here. The reported results are only for this dataset split and are not a guarantee of performance on new phones, notes, lighting, or physical banknotes.

In [ ]:
import hashlib, json
import numpy as np
import torch
from torch.utils.data import DataLoader
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix
missing_setup = [key for key in ('model', 'CropDataset', 'batch_size', 'WORK') if key not in globals()]
if missing_setup: raise RuntimeError(f'Missing {missing_setup}. Run training first, then rerun evaluation/export.')
test_loader = DataLoader(CropDataset('test'), batch_size=batch_size, shuffle=False, num_workers=0)
true_ids, predicted_ids = [], []
with torch.inference_mode():
    for images, labels in test_loader:
        logits = model(images)
        true_ids.extend(labels.numpy().tolist())
        predicted_ids.extend(logits.argmax(1).numpy().tolist())
accuracy = float(accuracy_score(true_ids, predicted_ids))
macro_f1 = float(f1_score(true_ids, predicted_ids, average='macro', zero_division=0))
print('Test accuracy:', accuracy, '| macro F1:', macro_f1)
print(classification_report(true_ids, predicted_ids, labels=list(range(6)), target_names=[f'PHP {d}' for d in DENOMS], zero_division=0))
print('Confusion matrix (rows=true, columns=predicted):\n', confusion_matrix(true_ids, predicted_ids, labels=list(range(6))))
manifest_sha = hashlib.sha256(manifest_path.read_bytes()).hexdigest()
checkpoint_sha = hashlib.sha256(CHECKPOINT.read_bytes()).hexdigest()
ONNX_PATH = WORK / 'peso_denomination_candidate.onnx'
torch.onnx.export(model, (torch.zeros(1,3,224,224),), str(ONNX_PATH), input_names=['images'], output_names=['logits'], dynamic_axes={'images':{0:'batch'},'logits':{0:'batch'}}, opset_version=17, dynamo=False)
import onnx, onnxruntime as ort
onnx.checker.check_model(onnx.load(str(ONNX_PATH)))
sample = np.random.default_rng(42).standard_normal((2,3,224,224), dtype=np.float32)
with torch.inference_mode(): torch_out = model(torch.from_numpy(sample)).numpy()
session = ort.InferenceSession(str(ONNX_PATH), providers=['CPUExecutionProvider'])
onnx_out = session.run(['logits'], {'images':sample})[0]
parity_error = float(np.max(np.abs(torch_out-onnx_out)))
if not np.allclose(torch_out, onnx_out, rtol=1e-3, atol=1e-4): raise RuntimeError(f'ONNX parity failed; max absolute error {parity_error}')
onnx_sha = hashlib.sha256(ONNX_PATH.read_bytes()).hexdigest()
metadata = {'model_present':True,'deployment_approved':False,'candidate_status':'experimental_unreviewed','approval_blocker':'Roboflow export has no physical-banknote IDs; image/source grouping cannot establish note-level leakage safety.','architecture':'torchvision.mobilenet_v3_small','task':'experimental genuine-source denomination classification only; not counterfeit detection or authenticity assessment','class_ids':list(DENOMS),'onnx_file':ONNX_PATH.name,'onnx_sha256':onnx_sha,'input_name':'images','output_name':'logits','input':{'shape':['batch',3,224,224],'layout':'NCHW','color':'RGB','preprocessing':'fit within 224x224 using Pillow aspect-preserving resize and centered black padding, then ImageNet mean/std normalization'},'normalization':{'mean':list(MEAN),'std':list(STD)},'test_evaluation':{'split':'test','accuracy':accuracy,'macro_f1':macro_f1,'manifest_sha256':manifest_sha,'checkpoint_sha256':checkpoint_sha},'onnxruntime_cpu_parity_max_abs_error':parity_error,'data_source':{'name':'Philippine Banknotes v8','license':'CC BY 4.0','url':'https://universe.roboflow.com/philippine-banknotes/philippine-banknotes/dataset/8','source_split_grouping':'Roboflow image stem; physical note identities unavailable'}}
META_PATH = WORK / 'model_metadata.json'
META_PATH.write_text(json.dumps(metadata, indent=2)+'\n')
print('ONNX exported and parity checked:', ONNX_PATH.name, '| max abs error:', parity_error)
print('Model is explicitly NOT approved for production/local scanner installation.')

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import json, shutil
missing_setup = [key for key in ('CHECKPOINT', 'ONNX_PATH', 'META_PATH', 'manifest_path', 'accuracy', 'macro_f1') if key not in globals()]
if missing_setup: raise RuntimeError(f'Missing {missing_setup}. Run evaluation and ONNX export first, then rerun packaging.')
RUN_BASE = Path('/content/drive/MyDrive/PESO_SCAN/artifacts') if DRIVE_MOUNTED else Path('/content/peso_scan_artifacts')
RUN_DIR = RUN_BASE / datetime.now(timezone.utc).strftime('roboflow-candidate-%Y%m%d-%H%M%S-%f')
RUN_DIR.mkdir(parents=True, exist_ok=True)
for artifact in (CHECKPOINT, ONNX_PATH, META_PATH, manifest_path): shutil.copy2(artifact, RUN_DIR / artifact.name)
metrics = {'test_accuracy':accuracy,'test_macro_f1':macro_f1,'source_crop_count':len(rows),'invalid_crop_count':len(invalid_crops),'source_image_group_count':len(group_splits),'source_split_overlap_groups':len(overlap),'physical_note_ids_available':False,'deployment_approved':False}
(RUN_DIR / 'candidate_metrics.json').write_text(json.dumps(metrics, indent=2)+'\n')
archive_path = Path(shutil.make_archive(str(RUN_DIR), 'zip', root_dir=RUN_DIR))
print('Candidate artifacts saved:', RUN_DIR)
if DRIVE_MOUNTED:
    print('Candidate archive saved in Drive:', archive_path)
else:
    from google.colab import files
    print('Drive was not mounted, so downloading the candidate archive to your computer now.')
    files.download(str(archive_path))
print('Keep this as an experimental candidate. Do not describe the score as authenticity confidence.')